In [3]:
from pathlib import Path
from pdfminer.high_level import extract_text as pdfminer_extract_text
from unstructured.chunking.title import chunk_by_title
from unstructured.cleaners.core import clean_extra_whitespace, clean_non_ascii_chars
from unstructured.partition.auto import partition
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.text import partition_text
from unstructured.documents.elements import Header, Footer

from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

import os
from dotenv import load_dotenv


In [4]:
# urls articles
faq_urls = [
    "https://teaspoonofadventure.com/75-questions-for-travellers/",
    "https://www.adventure-life.com/rwanda/articles/rwanda-faqs",
]

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR.parent / "itineraries" / "data"

pdf_paths = [
    str(DATA_DIR / "TravelTips-Oct2008.PDF"),
    str(DATA_DIR / "The-Best-100-Travel-Tips-and-Hacks-by-Jessica-Ufuoma-1.pdf"),
]

# Partition articles urls
def partition_article(urls):
    results = {}

    for url in urls:
        results[url] = partition(url=url)

    return results


# Partition pdf documents
def partition_pdf_documents(pdf_paths):
    results = {}

    for pdf_path in pdf_paths:
        elements = partition_pdf(filename=pdf_path, strategy="fast")

        if not elements:
            text = pdfminer_extract_text(pdf_path)
            elements = partition_text(text=text)

        results[pdf_path] = elements

    return results


# Clean the elements by removing empty or irrelevant text
def clean_elements(elements):
    cleaned = []

    for element in elements:
        if not element.text:
            continue

        text = clean_extra_whitespace(element.text)
        text = clean_non_ascii_chars(text)

        if not text.strip():
            continue

        element.text = text
        cleaned.append(element)

    return cleaned


# Filter unecessary elements from the documents and articles
def filter_elements(elements):
    filtered = []

    for element in elements:
        if not element.text:
            continue

        text = element.text.strip()

        if not text:
            continue

        # Remove very short fragments
        if len(text) < 30:
            continue

        # Remove headers and footers
        if isinstance(element, (Header, Footer)):
            continue

        filtered.append(element)

    return filtered


def preprocess_elements(elements):
    elements = clean_elements(elements)
    elements = filter_elements(elements)

    return elements


# save metadata for each source
def save_metadata(elements, source):
    for element in elements:
        element.metadata.filename = source
    return elements


def preprocess_and_chunk(elements, source):
    elements = save_metadata(elements, source)
    elements = clean_elements(elements)
    elements = filter_elements(elements)
    elements = chunk_by_title(
        elements,
        max_characters=4000,
        new_after_n_chars=3000,
        combine_text_under_n_chars=500,
    )

    return elements


articles = partition_article(faq_urls)
documents = partition_pdf_documents(pdf_paths)


chunked_articles = {
    source: preprocess_and_chunk(elements, source)
    for source, elements in articles.items()
}

chunked_documents = {
    source: preprocess_and_chunk(elements, source)
    for source, elements in documents.items()
}

No languages specified, defaulting to English.
No languages specified, defaulting to English.
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


In [5]:
# Embed the chunks using a sentence transformer model from HuggingFace

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4924.86it/s]


In [6]:
# Combine all chunks (articles and documents) 

all_chunks = []

for source, chunks in chunked_articles.items():
    for chunk in chunks:
        all_chunks.append(chunk)

for source, chunks in chunked_documents.items():
    for chunk in chunks:
        all_chunks.append(chunk)

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 43


In [7]:
# Get the text

texts = [chunk.text for chunk in all_chunks]

print(f"Number of texts: {len(texts)}")
print(texts[:2]) 

Number of texts: 43
['75 Questions Every Traveller Should Answer\n\nByRiana Ang-Canning November 1, 2018January 23, 2024\n\nIm going to keep things short and sweet because we have a lot of questions to get to! Read on for 75 questions every traveller should answer. Ive left my answers below but would love to hear yours!\n\nWhats your favourite place so far?\n\nThis is one of the toughest questions every traveller should answer. How do you pick just one? I always say that London is my favourite city in the world but I also love Pender Harbour, our annual family vacation spot.\n\nIf you could swim with dolphins or go shark diving, which would you pick?\n\nIve actually done both! I did a dolphin encounter at Atlantis in the Bahamas and went shark cave diving in Durban, South Africa. But both were in enclosed spaces with animals in partial-captivity. If I get the chance to do it in the wild, Ill pick the dolphins!\n\nWhat place is top of your bucket list?\n\nI hate to say it, but probably 

In [8]:
# Generating embeddings

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

print(embeddings.shape) 

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.98it/s]

(43, 384)


In [9]:
# save the metadata and embeddings to a file

embedded_documents = []

for chunk, embedding in zip(all_chunks, embeddings):
    embedded_documents.append({
        "text": chunk.text,
        "embedding": embedding,
        "metadata": chunk.metadata.to_dict(),
    })

In [10]:
print(f"Number of embedded documents: {len(embedded_documents)}")

embedded_documents[0]

Number of embedded documents: 43


{'text': '75 Questions Every Traveller Should Answer\n\nByRiana Ang-Canning November 1, 2018January 23, 2024\n\nIm going to keep things short and sweet because we have a lot of questions to get to! Read on for 75 questions every traveller should answer. Ive left my answers below but would love to hear yours!\n\nWhats your favourite place so far?\n\nThis is one of the toughest questions every traveller should answer. How do you pick just one? I always say that London is my favourite city in the world but I also love Pender Harbour, our annual family vacation spot.\n\nIf you could swim with dolphins or go shark diving, which would you pick?\n\nIve actually done both! I did a dolphin encounter at Atlantis in the Bahamas and went shark cave diving in Durban, South Africa. But both were in enclosed spaces with animals in partial-captivity. If I get the chance to do it in the wild, Ill pick the dolphins!\n\nWhat place is top of your bucket list?\n\nI hate to say it, but probably my phone or 

In [ ]:

# Create an index in pinecone

load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")


pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "travel-rag"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

In [12]:
index = pc.Index(index_name)  

In [13]:
# prepare the vectors for upsert
def prepare_vectors(embedded_documents):
    vectors = []

    for i, document in enumerate(embedded_documents):
        metadata = document["metadata"].copy()

        # Store the original text in metadata.
        metadata["text"] = document["text"]

        vectors.append({
            "id": f"travel-{i}",
            "values": document["embedding"].tolist(),
            "metadata": metadata,
        })

    return vectors

In [14]:
vectors = prepare_vectors(embedded_documents)

print(f"Prepared {len(vectors)} vectors")
# print(vectors[0:5])

Prepared 43 vectors


In [15]:
index.upsert(
    vectors=vectors,
    namespace="travel",
)

UpsertResponse(upserted_count=43)